# GEMM in Layout Algebra

This notebook walks through the **GEMM** algorithm and applications from Cecka,
*CuTe Layout Representation and Algebra*
([arXiv:2603.02298](https://arxiv.org/abs/2603.02298), §2.6.2),
implemented in PyCuTe.

Everything below calls one primitive,
[`pycute.alg.ref.gemm`](../../pycute/alg/ref/gemm.py):

```python
C[m,n,l] += A[m,k,l] * B[n,k,l]
```

1. **BLAS variants** — NT / TN / NN and the generically-strided BLIS GEMM are
   the same loop nest over different `A`, `B`, `C` layouts.
2. **Folding and GETT** — every tensor contraction becomes that loop nest by
   grouping its modes into row / column / reduction / batch multi-modes.
   `einsum` does the grouping; the applications of
   [`examples/einsum_test.py`](../einsum_test.py) are reviewed here.
3. **CONV** — the `im2col` transformation as a Layout, so the same loop nest
   computes an N-D convolution — with padding, dilation, traversal stride and
   `dgrad` — without moving any data. This is what CUTLASS calls *implicit
   GEMM*, and it is a change of view rather than a change of algorithm.

The CONV section uses
[`examples/im2col.py`](../im2col.py), whose tests are
[`examples/im2col_test.py`](../im2col_test.py).

In [1]:
import contextlib
import ctypes
import importlib.util
import inspect
import io
import itertools
import sys
import tempfile
from pathlib import Path

from IPython.display import SVG, display

# `pycute` and `examples` are plain source directories rather than installed
# packages, so a checkout without `pip install -e .` needs the root on the path.
if importlib.util.find_spec("examples") is None:
    for _root in (Path.cwd(), *Path.cwd().parents):
        if (_root / "examples" / "einsum.py").exists():
            sys.path.insert(0, str(_root))
            break

from examples.einfold import einfold
from examples.einsum import einsum, _classify, _fold, _parse
from examples.im2col import im2col, im2col_coord, im2col_padded, OutOfBoundsAccessor
from pycute import *
from pycute.alg.ref import gemm
from pycute.util import draw_svg, print_tensor

HAS_SVG = importlib.util.find_spec("svgwrite") is not None
if not HAS_SVG:
    print("svgwrite not installed; layout figures skipped. pip install -e '.[viz]'")

## Helpers

Demo tensors are `int64`, so every product below is exact and prints without
float noise. `make_tensor` allocates `coshape(layout)` zeroed elements, and
`tensor(layout, value)` then writes `value(i)` at each logical index `i`.

Wherever a figure wants a recognizable image, the fill goes through `coordinates`. 
An `im2col` layout is not injective, and reading/writing through one has several
logical positions land on the same physical value.

In [2]:
from pycute.util.draw_colors import index_grey_8x

_SVG_DIR = Path(tempfile.mkdtemp(prefix="pycute-gemm-"))
_SVG_COUNT = itertools.count()

def show(t, name=None, color=index_grey_8x):
    """
    Draw a Layout or Tensor, labelled `name`, falling back to a table when it
    cannot be drawn: `draw_svg` handles rank 1 and 2 only, writes a file rather
    than returning one, and a layout on basis strides maps to coordinates, which
    have no address to color.
    """
    layout = t.layout if isinstance(t, Tensor) else t
    if name is not None:
        print(f"{name}: {layout}")
    if HAS_SVG and rank(t) in (1, 2) and isinstance(layout(0), int):
        path = _SVG_DIR / f"layout{next(_SVG_COUNT)}.svg"
        with contextlib.redirect_stdout(io.StringIO()):
            draw_svg(t, filename=str(path), color=color)
        display(SVG(str(path)))
    else:
        print_tensor(t, print_type=False)


def tensor(layout, value=lambda i: 0):
    """
    A zeroed int64 Tensor over `layout`, with `value(i)` written at each 
    integral coordinate.
    """
    T = make_tensor(layout, dtype=ctypes.c_int64)
    for i in range(size(layout)):
        T[i] = value(i)
    return T


def row_major(shape_):
    """A compact layout over `shape_` with the last mode contiguous."""
    return Layout(shape_, tuple(reversed(prefix_product(tuple(reversed(shape_))))))

## The reference algorithm

`gemm` is a **rank-3 algorithm**: it iterates the row, column, reduction and
batch index spaces — `m`, `n`, `k`, `l` — and every operand access goes through
that operand's own layout. Nothing in the loop nest knows how `A`,
`B` or `C` are stored, or even what rank they *really* are — a 1-D coordinate
addresses a **multi-mode** (a group of modes) exactly as it addresses a single
mode, so the rank-3 signature constrains nothing but the mode semantics.

The rest of this notebook is a tour of what those layouts can be.

Rank-2 operands are accepted as the unbatched case — `gemm` appends the size-1
batch mode itself, which is a view and copies nothing — so most cells below pass
`(M,K)`, `(N,K)` and `(M,N)` tensors and never mention `L`.

In [3]:
print(inspect.getsource(gemm))

def gemm(A: Tensor, B: Tensor, C: Tensor) -> None:
  """
  Reference batched GEMM: `C[m,n,l] += A[m,k,l] * B[n,k,l]`.

  Each operand is rank-3, but a mode may be a multi-mode, so an operand of any
  rank participates once its modes are folded into these four roles -- row `M`,
  column `N`, reduction `K` and batch `L`.

  Rank-2 operands are the unbatched case.

  Pre-conditions:
    rank(A) == rank(B) == rank(C), and either 3 or 2
    size[0](A) == size[0](C)                  the row extent M
    size[0](B) == size[1](C)                  the column extent N
    size[1](A) == size[1](B)                  the reduction extent K
    size[2](A) == size[2](B) == size[2](C)    the batch extent L
  """
  # In the non-batched case, append a size-1 batch mode for convenience.
  if rank(A) == 2 and rank(B) == 2 and rank(C) == 2:
    A, B, C = (Tensor(T.accessor, make_layout([T.layout[0], T.layout[1], Layout(1, 0)])) for T in (A, B, C))
  if rank(A) != 3 or rank(B) != 3 or rank(C) != 3:
    raise

**Table 3** (Whitepaper) — applications that are all `gemm` with
different operand layouts:

| Application | A-Layout | B-Layout | C-Layout |
|---|---|---|---|
| NT GEMM | `(M,K):(1,lda)` | `(N,K):(1,ldb)` | `(M,N):(1,ldc)` |
| TN GEMM | `(M,K):(lda,1)` | `(N,K):(ldb,1)` | `(M,N):(1,ldc)` |
| NTT GEMM | `(N,K):(1,ldb)` | `(M,K):(1,lda)` | `(N,M):(1,ldc)` |
| BLIS GEMM | `(M,K):(dma,dka)` | `(N,K):(dnb,dkb)` | `(M,N):(dmc,dnc)` |
| GETT | `((M₁,M₂),K):((1,W),X)` | `(N,K):(K,1)` | `((M₁,M₂),N):((1,Y),Z)` |
| GETT | `(M,(K₁,K₂)):(1,(W,X))` | `(N,(K₁,K₂)):(Y,(1,Z))` | `(M,N):(1,M)` |
| CONV | `(K,(C,T,R,S)):D_A` | `((N,Z,P,Q),(C,T,R,S)):D_B` | `(K,(N,Z,P,Q)):D_C` |

The batched variants append one more mode, `L`, to each operand — grouped GEMM
is that mode with one index per group, and an operand *shared* across groups
takes stride `0` in it, so all `G` groups read one copy. Every row is just a
layout, so every row is the same four loops.

### BLAS transpose variants

The transpose flags of `xGEMM` are not an algorithm choice, they are a stride
choice. `A` is `(M,K)` and `B` is `(N,K)` *logically* in every variant; "N" and
"T" only say which of the two strides is `1`. BLIS drops even that constraint
and lets both strides be arbitrary.

Below, `A` and `B` hold the same logical values in all four cases, so the four
runs must agree on `C` while disagreeing completely on memory.

In [4]:
M, N, K = 4, 3, 2
lda, ldb, ldc = M, N, M

blas_cases = [
    ("NT",   Layout((M, K), (1, lda)),   Layout((N, K), (1, ldb)),   Layout((M, N), (1, ldc))),
    ("TN",   Layout((M, K), (K, 1)),     Layout((N, K), (K, 1)),     Layout((M, N), (1, ldc))),
    ("NN",   Layout((M, K), (1, lda)),   Layout((N, K), (K, 1)),     Layout((M, N), (N, 1))),
    ("BLIS", Layout((M, K), (2, 2 * M)), Layout((N, K), (3, 3 * N)), Layout((M, N), (2, 2 * M))),
]

value = lambda i, k: 1 + i + 10 * k     # the same logical values in every variant
expected = [[sum(value(m, k) * value(n, k) for k in range(K))
             for n in range(N)] for m in range(M)]

for label, la, lb, lc in blas_cases:
    A, B, C = tensor(la), tensor(lb), tensor(lc)
    for m, k in coordinates(shape(A)):   # fill by *logical* coordinate
        A[m, k] = value(m, k)
    for n, k in coordinates(shape(B)):
        B[n, k] = value(n, k)

    gemm(A, B, C)

    got = [[C[m, n] for n in range(N)] for m in range(M)]
    flat = Tensor(C.accessor, Layout(int(coshape(lc))))     # the buffer memory sees
    buffer = " ".join(f"{flat[i]}" if flat[i] else "." for i in range(size(flat)))
    print(f"{label:5} A {str(la):18} B {str(lb):18} C {str(lc):18} "
          f"{'match' if got == expected else 'DIFFER'}")
    print(f"{'':5} C over {int(coshape(lc)):2} addresses: {buffer}")

NT    A (4, 2):(1, 4)      B (3, 2):(1, 3)      C (4, 3):(1, 4)      match
      C over 12 addresses: 122 134 146 158 134 148 162 176 146 162 178 194
TN    A (4, 2):(2, 1)      B (3, 2):(2, 1)      C (4, 3):(1, 4)      match
      C over 12 addresses: 122 134 146 158 134 148 162 176 146 162 178 194
NN    A (4, 2):(1, 4)      B (3, 2):(2, 1)      C (4, 3):(3, 1)      match
      C over 12 addresses: 122 134 146 134 148 162 146 162 178 158 176 194
BLIS  A (4, 2):(2, 8)      B (3, 2):(3, 9)      C (4, 3):(2, 8)      match
      C over 23 addresses: 122 . 134 . 146 . 158 . 134 . 148 . 162 . 176 . 146 . 162 . 178 . 194


Same logical answer, four different buffers. `BLIS` is the interesting row: both
of its strides are non-unit, so `C` spans 23 addresses to hold 12 elements. A
BLAS `ld*` can stretch one of the two modes; a layout stretches both, and the
loop nest never notices.

---

## Folding — tensor contractions (GETTs) are batched GEMMs

The Whitepaper's example tensor contraction (§2.2) is

$$\mathbf{C}_{stqp} = \mathbf{A}_{stupr}\,\mathbf{B}_{qtru}$$

which is *already* a batched GEMM once its modes are sorted by where they
appear:

| Class | Appears in | Here |
|---|---|---|
| row `M̂` | `A`, `C` | `s`, `p` |
| column `N̂` | `B`, `C` | `q` |
| reduction `K̂` | `A`, `B` | `u`, `r` |
| batch `L̂` | `A`, `B`, `C` | `t` |

Grouping each class into one multi-mode rewrites the contraction as
$\mathbf{C}_{(sp)(q)(t)} = \mathbf{A}_{(sp)(ur)(t)}\,\mathbf{B}_{(q)(ur)(t)}$,
which is exactly `gemm`'s `(M,K,L) × (N,K,L) → (M,N,L)`. This is *tensor
folding*, and because a multi-mode is a layout of layouts it costs nothing: the
folded operands share the original accessor.

In [5]:
print(inspect.getsource(_classify))

def _classify(a_mode, b_mode, c_mode):
  """
  Sort the labels by which operands they appear in (Whitepaper, "Tensors and
  Folding"), returning `(row, col, red, bat)` in first-appearance order:

    row `M` -- A,C     col `N` -- B,C     red `K` -- A,B     bat `L` -- A,B,C

  Those four classes are the four index spaces of a batched GEMM, so this sort
  is the whole of what makes a contraction one. A label appearing in a single
  operand has no role among them and is rejected.
  """
  SA, SB, SC = set(a_mode), set(b_mode), set(c_mode)
  order = list(dict.fromkeys(a_mode + b_mode + c_mode))
  row = [x for x in order if x in SA and x in SC and x not in SB]
  col = [x for x in order if x in SB and x in SC and x not in SA]
  red = [x for x in order if x in SA and x in SB and x not in SC]
  bat = [x for x in order if x in SA and x in SB and x in SC]
  unsupported = (SA | SB | SC) - set(row + col + red + bat)
  if unsupported:
    raise ValueError(f"einsum: label(s) {sorted(unsupported)} app

In [6]:
extents = dict(s=2, t=3, u=2, p=2, r=3, q=2)
A = tensor(Layout(tuple(extents[c] for c in "stupr")), lambda i: i + 1)
B = tensor(Layout(tuple(extents[c] for c in "qtru")), lambda i: 2 * i + 3)
C = tensor(Layout(tuple(extents[c] for c in "stqp")))

A3 = _fold(A, "stupr", (["s", "p"], ["u", "r"], ["t"]))   # (M,K,L)
B3 = _fold(B, "qtru",  (["q"],      ["u", "r"], ["t"]))   # (N,K,L)
C3 = _fold(C, "stqp",  (["s", "p"], ["q"],      ["t"]))   # (M,N,L)

for name, flat, folded in (("A", A, A3), ("B", B, B3), ("C", C, C3)):
    print(f"{name} {str(flat.layout):40} -> {folded.layout}")
print(f"\nM = {size[0](A3)}   N = {size[0](B3)}   K = {size[1](A3)}   L = {size[2](A3)}")
print(f"same accessor, no copy: {A3.accessor == A.accessor}")

A (2, 3, 2, 2, 3):(1, 2, 6, 12, 24)        -> ((2, 2), (2, 3), 3):((1, 12), (6, 24), 2)
B (2, 3, 3, 2):(1, 2, 6, 18)               -> (2, (2, 3), 3):(1, (18, 6), 2)
C (2, 3, 2, 2):(1, 2, 6, 12)               -> ((2, 2), 2, 3):((1, 12), 6, 2)

M = 4   N = 2   K = 6   L = 3
same accessor, no copy: True


### Applications

`einsum(subscripts, A, B, C)` classifies subscript labels, folds the three operands,
and calls `gemm`. The table below is the application list from
[`examples/einsum_test.py`](../einsum_test.py), and every row runs the real
`einsum` — `_parse` and `_classify` are imported from it too, so the `M`, `N`,
`K`, `L` columns report what the implementation actually decided.

The one thing written here is `oracle`, which has to be independent to be worth
anything: it evaluates the contraction from its definition, with no layout
algebra involved.

In [7]:
def oracle(subscripts, extents, A, B):
    """
    `{c_coord: value}` for `C[c] = sum over the contracted labels of A*B`. One
    extent per label makes the contraction's index space a Shape, so its
    coordinates enumerate every term exactly once, with no layout involved.
    """
    a, b, c = _parse(subscripts)
    labels = list(dict.fromkeys(a + b + c))
    out = {}
    for combo in coordinates(tuple(extents[label] for label in labels)):
        at = dict(zip(labels, combo))
        ci = tuple(at[label] for label in c)
        term = A[tuple(at[label] for label in a)] * B[tuple(at[label] for label in b)]
        out[ci] = out.get(ci, 0) + term
    return out


def run(subscripts, extents):
    """Build the operands for `subscripts`, run `einsum`, check the oracle."""
    a, b, c = _parse(subscripts)
    def make(mode, value=lambda i: 0):
        return tensor(Layout(tuple(extents[ch] for ch in mode)), value)
    A = make(a, lambda i: i + 1)
    B = make(b, lambda i: 2 * i + 3)
    C = make(c)
    einsum(subscripts, A, B, C)
    return all(C[ci] == v for ci, v in oracle(subscripts, extents, A, B).items())


APPLICATIONS = [
    ("mk,nk->mn",        dict(m=3, n=4, k=5),                "GEMM"),
    ("mk,nk->nm",        dict(m=3, n=4, k=5),                "GEMM, transposed output"),
    ("bmk,bnk->bmn",     dict(b=2, m=3, n=4, k=5),           "batched (and grouped) GEMM"),
    ("mkb,nkb->mnb",     dict(b=2, m=3, n=4, k=5),           "batched GEMM, batch last"),
    ("ijp,np->ijn",      dict(i=2, j=3, p=4, n=5),           "GETT, two row modes"),
    ("stupr,qtru->stqp", dict(s=2, t=3, u=2, p=2, r=3, q=2), "GETT, Whitepaper contraction"),
    ("i,i->",            dict(i=6),                          "inner product"),
    ("i,j->ij",          dict(i=3, j=4),                     "outer product"),
    ("i,i->i",           dict(i=6),                          "Hadamard product"),
    ("ij,ij->ij",        dict(i=3, j=4),                     "Hadamard product, 2-D"),
    ("i,->i",            dict(i=5),                          "scalar scaling"),
    ("mn,->nm",          dict(m=5, n=6),                     "scalar scaling + transpose"),
    ("ij,j->i",          dict(i=3, j=4),                     "matrix-vector"),
    ("i,ij->j",          dict(i=3, j=4),                     "vector-matrix"),
    ("bij,bj->bi",       dict(b=2, i=3, j=4),                "batched matrix-vector"),
]

print(f"{'subscripts':18} {'M':>5} {'N':>5} {'K':>5} {'L':>5}   {'':8} note")
for subscripts, extents, note in APPLICATIONS:
    row, col, red, bat = _classify(*_parse(subscripts))
    def j(labels):
        return "".join(labels) or "-"
    print(f"{subscripts:18} {j(row):>5} {j(col):>5} {j(red):>5} {j(bat):>5}   "
          f"{'match' if run(subscripts, extents) else 'DIFFER':8} {note}")

subscripts             M     N     K     L            note
mk,nk->mn              m     n     k     -   match    GEMM
mk,nk->nm              m     n     k     -   match    GEMM, transposed output
bmk,bnk->bmn           m     n     k     b   match    batched (and grouped) GEMM
mkb,nkb->mnb           m     n     k     b   match    batched GEMM, batch last
ijp,np->ijn           ij     n     p     -   match    GETT, two row modes
stupr,qtru->stqp      sp     q    ur     t   match    GETT, Whitepaper contraction
i,i->                  -     -     i     -   match    inner product
i,j->ij                i     j     -     -   match    outer product
i,i->i                 -     -     -     i   match    Hadamard product
ij,ij->ij              -     -     -    ij   match    Hadamard product, 2-D
i,->i                  i     -     -     -   match    scalar scaling
mn,->nm               mn     -     -     -   match    scalar scaling + transpose
ij,j->i                i     -     j     -   match    

Read the dashes as size-1 modes. An inner product `i,i->` has no row, column or
batch mode at all: `M = N = L = 1` and the whole computation is the `k` loop.
An outer product `i,j->ij` is the mirror image, `K = 1`. A Hadamard product
`i,i->i` puts its only label in the *batch* class, so it runs as `size(i)`
independent `1×1×1` GEMMs. None of these are special cases in the
implementation; they are what the classification returns.

`mn,->nm` is the extreme: `B` is rank-0, every label is a row mode, and the
result is a scaled transpose.

---

## CONV — convolution as implicit GEMM

A convolution applies a stencil to every position of an activation. Written
directly it is a deep loop nest with not much that looks like a matrix
product:

```
y[n,k,z,p,q] = Σ_{c,t,r,s} w[k,t,r,s,c] · x[n, u·z + a·t, u·p + a·r, u·q + a·s, c]
```

But sort those indices by which tensors they appear in, exactly as the folding
section did, and the four GEMM roles appear:

| Class | Appears in | Convolution name |
|---|---|---|
| row `M̂` | activation, output | `n`, `z`, `p`, `q` — the output positions |
| column `N̂` | filter, output | `k` — the output channels |
| reduction `K̂` | activation, filter | `t`, `r`, `s`, `c` — the stencil volume |
| batch `L̂` | — | none |

So convolution *is* a GEMM of shape `(N·Z·P·Q) × (K) × (T·R·S·C)`. The one
difficulty is the activation: its reduction modes `t,r,s` and its row modes
`z,p,q` **index the same axes of the same tensor**, added together. The Layout
will not be injective.

Whitepaper Table 3's CONV row is that GEMM, transposed so that the activation
is `A` — the operand the rest of this section is about:

| | Shape | |
|---|---|---|
| `A` activation | `((N,(Z,P,Q)), ((T,R,S),C))` | output positions × stencil volume |
| `B` filter | `(K, ((T,R,S),C))` | output channels × stencil volume |
| `C` output | `((N,(Z,P,Q)), K)` | output positions × output channels |

Two notes. Convolution's `K` (output channels) is the GEMM's `N`, and
convolution's `C·T·R·S` is the GEMM's `K`; the collision is inherited from both
literatures. And the reduction mode is grouped
`((T,R,S),C)`, where CUTLASS example 59 groups the same modes as `(C,(T,R,S))`;
either order is fine so long as the `A` and `B` operands agree.

`B` and `C` are ordinary folds — `einfold` writes them in one string each. All
the im2col work is in transforming `A`.

### `im2col`, traditionally: a copy

The textbook way to get that transformed `A`-matrix is to build it. Gather each sliding window of
the activation into a row of a new matrix:

```python
def im2col(signal, Z, T):                  # signal = [0, 1, 2, 3, 4, 5], T = 3
    return [[signal[z + t] for t in range(T)] for z in range(Z)]

[[0, 1, 2],    <- signal[0:3]
 [1, 2, 3],    <- signal[1:4]
 [2, 3, 4],    <- signal[2:5]
 [3, 4, 5]]    <- signal[3:6]
```

A six-element signal is transformed into a twelve element matrix. No arithmetic is done, the
data is only rearranged.

Look at what that function actually is, though. It maps a pair — output
position `z`, stencil tap `t` — to an offset into the activation, `z + t`.

**That is what a Layout is.** So there is no need to build the matrix; the map
from coordinates to offsets alone will do.

### 1-D convolution, end to end

Take a six-element signal (1 batch `N` and 1 channel `C` for simplicity) and a three-tap stencil.

The view we want has two modes, each derived from that one spatial mode of the signal:

* **mode 1, the window** `T` — which offsets make up a single stencil tap window;
* **mode 0, the origins** `Z` — where each window starts.

Each mode is a layout in its own right, so each can be evaluated and drawn over the
signal's addresses. Together they span the `(Z,T)` matrix whose entry at `(z,t)`
is the offset `z + t`, an origin plus a tap — that sum is the whole of `im2col`,
and it is why the matrix holds 12 entries over 6 addresses.

The im2col view is the GEMM's `A`-matrix, the filter bank its `B`-matrix and the response its `C`-matrix.
Storing each in the shape the contraction wants — `((N,Z),(T,C))` for the
im2col `A`-matrix activations, `(K,(T,C))` for the `B`-matrix filter, and
`((N,Z),K)` for the `C`-matrix response — lets one `gemm` call compute the
convolution.

In [8]:
N = 1                                                         # The number of batches
C = 1                                                         # The number of channels
D = 6                                                         # A six-element signal
T = 3                                                         # A three-element stencil
Z = D - T + 1                                                 # The number of stencils

act = tensor(Layout((N, D, C)))                               # (N,D,C), still zeroed
A_mk = im2col(act, stencil=T)                                 # ((N,Z),(T,C)) a view, not a copy

def mark(offsets, on=(118, 185, 0), off=(246, 248, 244)):
    """A coloring that marks the cells reading `offsets` and mutes the rest."""
    marked = {int(o) for o in offsets}
    return lambda idx: on if int(idx) in marked else off

print(f"activation {act.layout}      im2col {A_mk.layout}\n")
show(Layout(D), "the signal's addresses")

for label, mode in (("mode 1, the window ", A_mk.layout[1]),
                    ("mode 0, the origins", A_mk.layout[0])):
    offsets = [mode(i) for i in range(size(mode))]
    print(f"{label} {str(mode):22} reads offsets {offsets}")
    show(Layout(D), color=mark(offsets))

print(f"together: {size(A_mk)} offsets z + t over {int(coshape(A_mk.layout))} addresses")
show(A_mk.layout, "im2col")

SyntaxError: '(' was never closed (1764510877.py, line 7)

In [ ]:
for i in range(D):
    act[0, i, 0] = i                   # element i holds value i

taps = [[1, 0, -1],    # filter 0: edge detector
        [1, 1,  1]]    # filter 1: box filter
KF = len(taps)         # filter count -- the GEMM's N

flt = tensor(Layout((KF, (T, C))))                            # (    K, (T,C))
for k, (t, c) in coordinates(shape(flt)):
    flt[k, (t, c)] = taps[k][t]
out = tensor(Layout(((N, Z), KF)))                            # ((N,Z),     K)

gemm(A_mk,             # ((N,Z), (T,C))
     flt,              # (    K, (T,C))
     out)              # ((N,Z),     K)

got = [[out[(0, z), k] for z in range(Z)] for k in range(KF)]
want = [[sum(taps[k][t] * (z + t) for t in range(T)) for z in range(Z)]
        for k in range(KF)]
print(f"responses per filter {got}")
print("match" if got == want else "DIFFER")

responses per filter [[-2, -2, -2, -2], [3, 6, 9, 12]]
match


### From 1-D to N-D — compose the spatial modes twice

The rule that generalizes is short. Take all of the activation's spatial modes,
`A[1:-1]` of a `(N, D, [H,] [W,] C)` tensor — excluding the batch mode `N` and
the channel mode `C` — and **compose** each with a tiler, once for each half of
the result:

* with `Layout(Z, u)` per spatial mode, `u` the traversal stride, to get the
  output-position modes `(Z,P,Q)`;
* with `Layout(T, a)` per spatial mode, `a` the dilation, to get the stencil-tap
  modes `(T,R,S)`.

Both halves compose the *same* sublayout, and that is the whole of `im2col`: a
spatial coordinate comes out as `u·z + a·t` because both modes walk one axis.
That shared axis is also why the result is not injective — each element is read
once per window containing it. `N` and `C` are carried across untouched, and the
origin is shifted to the lower corner, which is where padding will live.

Below, a `4×4` image and a `2×2` stencil. Built by hand it is exactly what
`im2col` returns: `M = 3·3 = 9` windows of `K = 2·2 = 4` taps, with the
activation's spatial strides `(4, 1)` appearing in *both* halves. Cells are
colored by the address they read, so the `im2col` matrix carries the
activation's own sixteen colors — it *is* those elements, gathered.

In [ ]:
N, C = 1, 1                                             # The batch and channel sizes
D, H = 4, 4                                             # The spatial mode sizes
T, R = 2, 2                                             # The stencil sizes
Z, P = D - T + 1, H - R + 1                             # The number of stencils
img = tensor(row_major((1, D, H, 1)))                   # (N,D,H,C)
for n, d, h, c in coordinates(shape(img)):              # fill by coordinate
    img[n, d, h, c] = d * H + h

L = img.layout
A_dh = L[1:-1]                                         # the spatial modes, as one Layout
pos = composition(A_dh, (Layout(Z, 1), Layout(P, 1)))  # position composition, traversal stride 1
tap = composition(A_dh, (Layout(T, 1), Layout(R, 1)))  # stencil composition, dilation 1

print(f"{'spatial sublayout A[1:-1]':32} {A_dh}")
print(f"{'composed with the position tiler':32} {pos}   <- (Z,P)")
print(f"{'composed with the tap tiler':32} {tap}   <- (T,R)")

by_hand = make_layout([make_layout([L[0], pos]),              # (N,(Z,P))
                       make_layout([tap, L[-1]])])            # ((T,R),C)
B = im2col(img, stencil=(T, R))
print(f"\nby hand {by_hand}")
print(f"im2col  {B.layout}")
print(f"identical: {str(by_hand) == str(B.layout)}")
print(f"\na tiler with stride 2 doubles what it composes -- the traversal stride in "
      f"one half,\nthe dilation in the other: "
      f"{composition(A_dh, (Layout(2, 2), Layout(2, 2)))}")

show(Tensor(img.accessor, img.layout[1:-1]), "activation")   # its own spatial modes
show(B, "im2col view")

reads = {}
crd = im2col_coord(img, stencil=(T, R))
for i in range(size(crd)):
    coord = idx2crd(crd[i], shape(img))
    reads[coord] = reads.get(coord, 0) + 1

print(f"M = {size[0](B)} windows, K = {size[1](B)} taps")
print(f"size {size(B)} reads over coshape {int(coshape(B.layout))} elements "
      f"-- {size(B) / int(coshape(B.layout)):.2f}x amplification, materialized nowhere")
print("\nreads per pixel:")
for d in range(D):
    print("   ", " ".join(str(reads[(0, d, h, 0)]) for h in range(H)))

NameError: name 'tensor' is not defined

### Traversal stride, dilation, and the output extent

The two tilers carry the two scales, and they act on disjoint halves of the
result:

* `stride_dhw`, the **traversal stride**, scales the *position* strides — the
  windows start further apart, and there are fewer of them;
* `stride_trs`, the **dilation**, scales the *tap* strides — the stencil's
  footprint opens holes, and the window covers more ground.

Both change the output extent, through one formula per spatial mode. With `T`
the stencil extent, `p_lo`/`p_hi` the padding, `u` the traversal stride and `a`
the dilation:

$$Z = \left\lfloor \frac{D + p_{lo} + p_{hi} - ((T-1)\cdot a + 1)}{u} \right\rfloor + 1$$

which counts the positions at which a dilated stencil of `T` taps fits inside
the padded activation, taking every `u`-th one. A stencil that does not fit
raises rather than producing an empty or negative extent.

Each parameter carries one entry per spatial mode, in the activation's order,
and nothing is broadcast: `None` takes the default at every mode, and a rank
mismatch is an error rather than a guess.

In [ ]:
A7 = tensor(row_major((1, 7, 7, 1)))

print("one 7x7 activation, one 3x3 stencil, three ways:")
for label, kw in (("plain", {}),
                  ("traversal stride 2", dict(stride_dhw=(2, 2))),
                  ("dilation 2", dict(stride_trs=(2, 2)))):
    B7 = im2col(A7, (3, 3), **kw)
    print(f"  {label:20} {str(B7.layout):58} M = {size[0](B7):2}")

A5 = tensor(row_major((1, 5, 5, 1)))
print("\noutput extents:")
for label, T5, kw in (("5x5, 3x3", A5, {}),
                      ("+ padding (1,1)", A5, dict(lower_dhw=(1, 1))),
                      ("+ asymmetric (1,0)/(0,1)", A5, dict(lower_dhw=(1, 0), upper_dhw=(0, 1))),
                      ("7x7, traversal 2", A7, dict(stride_dhw=(2, 2))),
                      ("7x7, dilation 2", A7, dict(stride_trs=(2, 2)))):
    print(f"  {label:26} {shape(im2col(T5, (3, 3), **kw))[0][1]}")

for label, kw in (("a stencil that does not fit", dict(stride_trs=(3, 3))),
                  ("a scalar for two spatial modes", dict(stride_dhw=2))):
    try:
        im2col(A5, (3, 3), **kw)
    except ValueError as e:
        print(f"\n{label}:\n  {e}")

one 7x7 activation, one 3x3 stencil, three ways:
  plain                ((1, (5, 5)), ((3, 3), 1)):((49, (7, 1)), ((7, 1), 1))     M = 25
  traversal stride 2   ((1, (3, 3)), ((3, 3), 1)):((49, (14, 2)), ((7, 1), 1))    M =  9
  dilation 2           ((1, (3, 3)), ((3, 3), 1)):((49, (7, 1)), ((14, 2), 1))    M =  9

output extents:
  5x5, 3x3                   (3, 3)
  + padding (1,1)            (5, 5)
  + asymmetric (1,0)/(0,1)   (4, 4)
  7x7, traversal 2           (3, 3)
  7x7, dilation 2            (3, 3)

a stencil that does not fit:
  im2col: stencil (3, 3) dilated by (3, 3) does not fit in (5, 5) padded by (0, 0)/(0, 0): extent (-1, -1)

a scalar for two spatial modes:
  im2col: stride_dhw has rank 1 but expected 2


### Padding lives in the origin, and out-of-bounds in the accessor

Padding does not change the construction. The first window starts `lower_dhw`
before the activation's origin, so the whole codomain shifts, and the shift is
found by evaluating the activation's own layout at that corner coordinate. It
belongs to the accessor, not to the layout.

For the memory view that corner is a *negative offset*: the accessor sits before
the activation and reads whatever is there. The layout is right — it just names
memory the activation does not own. A Layout maps coordinates to offsets and
cannot return "zero", so this is the one place `im2col` leaves the algebra.

What rescues it is that the construction never inspects a stride, so it can be
run over a different codomain. On basis strides instead of the activation's
integers, the same layout maps to **coordinates** of the activation — and a
coordinate can be bounds-checked *before* anything is read. Three views follow
from one construction:

| view | codomain | for |
|---|---|---|
| `im2col` | offsets into memory | GEMM, when nothing is padded |
| `im2col_coord` | `(n,d,h,w,c)` coordinates | predication, inspection, TMA descriptors |
| `im2col_padded` | coordinates, bounds-checked, read | padded CONV through an unmodified GEMM |

`im2col_padded` is `im2col_coord`'s layout behind `OutOfBoundsAccessor`, so
padding costs one comparison per element and no memory at all — what TMA's
out-of-bounds fill does in hardware. It is read-only: a write to a padded
position has nowhere to go. And `im2col_coord` is exactly the
`ArithmeticTupleIterator` an SM90 TMA im2col descriptor is built from: the same
construction over a different stride algebra.

In [ ]:
A4 = tensor(row_major((1, 4, 4, 1)))

print(f"{'view':16} {'accessor':22} layout")
for label, T4 in (("im2col", im2col(A4, (2, 2))),
                  ("im2col_coord", im2col_coord(A4, (2, 2))),
                  ("im2col_padded", im2col_padded(A4, (2, 2)))):
    print(f"{label:16} {type(T4.accessor).__name__:22} {T4.layout}")

print("\na 3x3 stencil with one element of padding, corner and centre window:")
crd = im2col_coord(A4, (3, 3), lower_dhw=(1, 1))
for label, position in (("corner", ((0, (0, 0)), ((0, 0), 0))),
                        ("centre", ((0, (1, 1)), ((1, 1), 0)))):
    coord = crd[position]
    print(f"  {label}  activation coordinate {str(idx2crd(coord, shape(A4))):16} "
          f"in_bounds {in_bounds(coord, shape(A4))}")

offsets = im2col(A4, (3, 3), lower_dhw=(1, 1))
shift = (offsets.accessor.address - A4.accessor.address) // ctypes.sizeof(ctypes.c_int64)
print(f"\nthe memory view's origin is at offset {shift}: the corner (0,-1,-1,0) through")
print(f"strides {stride(A4.layout)}, which is memory the activation does not own.")
print("The accessor that bounds-checks instead:\n")
print(inspect.getsource(OutOfBoundsAccessor.__getitem__))

nine = tensor(Layout((1, 3, 3, 1)))
for n, d, h, c in coordinates(shape(nine)):
    nine[n, d, h, c] = 1 + d * 3 + h                 # 1..9, row-major, by coordinate

padded = im2col_padded(nine, (3, 3), lower_dhw=(1, 1))
show(padded, "padded im2col of a 3x3 image, 3x3 stencil, pad 1")

crd = im2col_coord(nine, (3, 3), lower_dhw=(1, 1))
fill = sum(1 for i in range(size(crd)) if not in_bounds(crd[i], shape(nine)))
print(f"{fill} of {size(crd)} cells are fill ({100 * fill / size(crd):.0f}%), and not one is stored")

### CONV as one `gemm`

Everything is now in place. `conv` below builds the three operands and makes a
single `gemm` call, for any number of spatial modes:

* `A` is the `im2col` view — `im2col_padded` when anything is padded, so that
  the out-of-bounds taps read zero;
* `B` is the filter `(K,*TRS,C)` folded to `(K, ((T,R,S),C))`, one `einfold`
  string;
* `C` is the output `(N,*ZPQ,K)` folded to `((N,(Z,P,Q)), K)`, likewise.

The two folds are built from the mode names, so `"ktc -> k((t)c)"` in 1-D
becomes `"ktrsc -> k((trs)c)"` in 3-D and nothing else changes.
`pycute.alg.ref.gemm` is untouched: it sees a rank-2 `(M,K)` tensor, contracts
it, and the convolution falls out.

`conv_reference` is the oracle — a direct N-D cross-correlation that reuses none
of the layout algebra, treating any activation coordinate outside the tensor as
zero.

In [ ]:
TAPS, POSITIONS = "trs", "zpq"          # a rank-n activation uses the first n


def conv(act, flt, stencil, **kw):
    """
    One `gemm` convolution of `act` (N,*DHW,C) by `flt` (K,*TRS,C) -> (N,*ZPQ,K).

    `kw` is `im2col`'s: `lower_dhw`, `upper_dhw`, `stride_dhw`, `stride_trs`.
    Any padding routes through `im2col_padded`, whose accessor reads zero
    outside the activation.
    """
    num_spatial = rank(act) - 2
    taps, positions = TAPS[:num_spatial], POSITIONS[:num_spatial]

    view = im2col_padded if kw.get("lower_dhw") or kw.get("upper_dhw") else im2col
    A = view(act, stencil, **kw)                                  # ((N,(Z,P,Q)), ((T,R,S),C))
    out = tensor(Layout((size[0](act), *shape(A)[0][1], size[0](flt))))

    gemm(A,
         einfold(f"k{taps}c -> k(({taps})c)", flt),               # (K, ((T,R,S),C))
         einfold(f"n{positions}k -> (n({positions}))k", out))
    return out, A


def conv_reference(act, flt, stencil, *, lower_dhw=None, upper_dhw=None,
                   stride_dhw=None, stride_trs=None):
    """
    `{(n,*zpq,k): value}` for a direct cross-correlation of `act` by `flt`.

    Activation coordinates outside `act` contribute nothing, which is zero
    padding. No layout algebra is used, so this is an independent check.
    """
    num_spatial = rank(act) - 2
    stencil = wrap(stencil)
    per = lambda v, default: tuple(v) if v is not None else (default,) * num_spatial
    lower = per(lower_dhw, 0)
    upper = lower if upper_dhw is None else tuple(upper_dhw)
    traversal, dilation = per(stride_dhw, 1), per(stride_trs, 1)

    shape_dhw, C, K = shape(act)[1:-1], shape(act)[-1], size[0](flt)
    zpq = tuple(1 + (d + lo + hi - ((t - 1) * a + 1)) // u
                for d, lo, hi, t, a, u in
                zip(shape_dhw, lower, upper, stencil, dilation, traversal))

    out = {}
    for n, k in coordinates((size[0](act), K)):
        for pos in coordinates(zpq):
            total = 0
            for tap in coordinates(stencil):
                crd = tuple(u * p + a * t - lo for p, t, u, a, lo in
                            zip(pos, tap, traversal, dilation, lower))
                if all(0 <= c < d for c, d in zip(crd, shape_dhw)):
                    for c in range(C):
                        total += act[(n, *crd, c)] * flt[(k, *tap, c)]
            out[(n, *pos, k)] = total
    return out

In [ ]:
CASES = [
    ("1-D",               (1, 6, 1),       (2, 3, 1),       (3,),      {}),
    ("2-D",               (1, 4, 4, 1),    (1, 2, 2, 1),    (2, 2),    {}),
    ("multi-channel",     (1, 5, 6, 3),    (4, 2, 3, 3),    (2, 3),    {}),
    ("batched",           (2, 5, 6, 3),    (4, 2, 3, 3),    (2, 3),    {}),
    ("traversal stride",  (2, 7, 7, 2),    (3, 3, 3, 2),    (3, 3),    dict(stride_dhw=(2, 2))),
    ("dilation",          (1, 7, 7, 2),    (2, 3, 3, 2),    (3, 3),    dict(stride_trs=(2, 2))),
    ("same padding",      (1, 5, 5, 2),    (3, 3, 3, 2),    (3, 3),    dict(lower_dhw=(1, 1))),
    ("asymmetric pad",    (1, 5, 5, 1),    (2, 3, 3, 1),    (3, 3),    dict(lower_dhw=(1, 0),
                                                                            upper_dhw=(0, 1))),
    ("1x1 -- a GEMM",     (1, 8, 8, 4),    (4, 1, 1, 4),    (1, 1),    {}),
    ("3-D, mixed",        (2, 4, 5, 6, 2), (3, 2, 3, 2, 2), (2, 3, 2), dict(lower_dhw=(0, 1, 1),
                                                                            stride_dhw=(1, 2, 1),
                                                                            stride_trs=(1, 1, 2))),
]

print(f"{'case':18} {'activation':17} {'stencil':10} {'M':>5} {'N':>4} {'K':>4}   result")
for label, shape_act, shape_flt, stencil, kw in CASES:
    act = tensor(row_major(shape_act), lambda i: i % 7 - 3)
    flt = tensor(row_major(shape_flt), lambda i: i % 5 - 2)
    out, A = conv(act, flt, stencil, **kw)
    if label.startswith("3-D"):
        mixed = A.layout                 # the one view the commentary below reads
    want = conv_reference(act, flt, stencil, **kw)
    ok = all(out[coord] == value for coord, value in want.items())
    print(f"{label:18} {str(shape_act):17} {str(stencil):10} {size[0](A):5} "
          f"{size[0](flt):4} {size[1](A):4}   {'match' if ok else 'DIFFER'}")

print(f"\nthe 3-D row is padded, so its view maps to coordinates:\n  {mixed}")
print("  each basis coefficient is that axis' scale, in the half that carries it:")
print("    position half  1@1 depth, 2@2 height   <- the traversal strides (1, 2, 1)")
print("    tap half       1@2 height, 2@3 width   <- the dilations         (1, 1, 2)")
print("  depth is plain, so 1@1 appears in both halves")

case               activation        stencil        M    N    K   result
1-D                (1, 6, 1)         (3,)           4    2    3   match
2-D                (1, 4, 4, 1)      (2, 2)         9    1    4   match
multi-channel      (1, 5, 6, 3)      (2, 3)        16    4   18   match
batched            (2, 5, 6, 3)      (2, 3)        32    4   18   match
traversal stride   (2, 7, 7, 2)      (3, 3)        18    3   18   match
dilation           (1, 7, 7, 2)      (3, 3)         9    2   18   match
same padding       (1, 5, 5, 2)      (3, 3)        25    3   18   match
asymmetric pad     (1, 5, 5, 1)      (3, 3)        16    2    9   match
1x1 -- a GEMM      (1, 8, 8, 4)      (1, 1)        64    4    4   match
3-D, mixed         (2, 4, 5, 6, 2)   (2, 3, 2)    108    3   24   match

the 3-D row is padded, so its view maps to coordinates:
  ((2, (3, 3, 6)), ((2, 3, 2), 2)):((1@0, (1@1, 2@2, 1@3)), ((1@1, 1@2, 2@3), 1@4))
  each basis coefficient is that axis' scale, in the half that car

Ten convolutions, one `gemm` each, no data movement anywhere.

The `3-D, mixed` row treats each axis differently and independently — depth
plain, height traversal-strided by two, width padded and dilated by two — and it
is still one call and one layout, with each axis' scale legible in the half of
the result that carries it. Because that case is padded it runs through
`im2col_padded`, so its layout is the coordinate one: the scales appear as basis
coefficients like `2@2` rather than as multiplied integer strides, which is if
anything easier to read back.

The `1×1` row is the degenerate case. With `T = R = S = 1` every tap tiler is
`Layout(1, ·)`, so the tap modes collapse to size 1 and `im2col` reduces to the
activation itself. That is not a special case in the code; it is what
the algebra returns when handed a one-tap stencil.

### `dgrad` — the same layout, walked backwards

The backward-data pass of a convolution reads the stencil in reverse. Two
parameters cover it, without a second construction. The spatial coordinate a
view reads is

$$\texttt{stride\_dhw}\cdot z \;+\; \texttt{stride\_trs}\cdot t \;+\; \texttt{lower\_trs} \;-\; \texttt{lower\_dhw}$$

so `fprop` leaves `lower_trs` at `0`, while `dgrad` sets
`lower_trs = (T-1)·dilation` and `stride_trs = -dilation` to walk the taps
backwards from the far end. `shape_zpq` supplies the output extent, since the
forward formula is not the one `dgrad` wants.

In [ ]:
sig = tensor(Layout((1, 6, 1)))                      # (N,D,C)
T = 3

d = im2col_coord(sig, T, lower_dhw=T - 1, lower_trs=T - 1, stride_trs=-1, shape_zpq=6)
text = str(d.layout)
print(text)
print(" " * text.index("-1@1") + "^^^^^ negative dilation\n")

print("the activation coordinate this view reads, per (z,t):")
for z in range(6):
    print("   ", [idx2crd(d[(0, (z,)), ((t,), 0)], shape(sig))[1] for t in range(T)])
print("\nrow z, column t reads coordinate z - t: the reversed correlation dgrad performs")

((1, (6,)), ((3,), 1)):((1@0, (1@1,)), ((-1@1,), 1@2))
                                         ^^^^^ negative dilation

the activation coordinate this view reads, per (z,t):
    [0, -1, -2]
    [1, 0, -1]
    [2, 1, 0]
    [3, 2, 1]
    [4, 3, 2]
    [5, 4, 3]

row z, column t reads coordinate z - t: the reversed correlation dgrad performs


### Implicit GEMM

A materialized `im2col` pays the amplification in memory and bandwidth:

| | `4×4`, `2×2`, `C=1` | `56×56`, `3×3`, `C=64`, pad 1 |
|---|---|---|
| activation | 16 | 200 704 |
| `im2col` matrix | 36 (**2.25×**) | 1 806 336 (**9×**) |

The layout pays it nowhere. The reads still happen — a nine-tap stencil still
reads each interior pixel nine times — but they hit cache rather than a second
buffer, and no allocation, no copy kernel and no extra bandwidth is involved.

The Tensors `im2col` and its variants generate can be tiled, partitioned,
predicated and manipulated in all the same ways as any other tensor. The GEMM
implementation need not know anything about CONV, and can be optimized
independently, or with a special dispatch (like im2col-TMA) for Layouts that
look like this. The convolution is a change of view, not a change of algorithm,
and the change of view is nine lines of layout algebra.

## Next

* An "optimized" GEMM to sit beside `pycute.alg.copy`, with algebraic operations
  tiling/partitioning the input tensors and lowering to another GEMM.
* `wgrad`, the remaining convolution gradient. Like `dgrad` it should be a
  permutation of the same three tensors' roles rather than new machinery.